# U-Net results — training curves and predictions

- Loss and mIoU curves (parsed from the slurm log)
- Visualization: RGB | ground-truth mask | prediction, on validation tiles.


In [ ]:
# Config (adjust if needed)
import os

LOG_PATH    = "slurm_seg_95458.log"
TILES_ROOT  = "/share/castor/home/e2406749/thesis_tiles_120px"
EMB_ROOT    = "embeddings"
SHP_PATH    = "data_shp/label_polygons.shp"
MODEL_PATH  = "seg_model_best.pt"

import re
import glob
import random
import numpy as np
import torch
import rasterio
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

from seg_dataset import DisturbanceSegDataset, NUM_CLASSES, CLASS_TO_ID
from seg_model import SegDecoder

ID_TO_CLASS = {v: k for k, v in CLASS_TO_ID.items()}
ID_TO_CLASS[0] = "background"
print("NUM_CLASSES:", NUM_CLASSES)

## A) Training curves (loss + mIoU per epoch)

If the loss drops but the mIoU plateaus -> overfitting.

In [ ]:
# Parse the Slurm log
epochs, losses, mious = [], [], []
with open(LOG_PATH) as f:
    for line in f:
        m = re.search(r"epoch\s+(\d+)\s+\|\s+loss\s+([\d.]+)\s+\|\s+mIoU\s+([\d.]+)", line)
        if m:
            epochs.append(int(m.group(1)))
            losses.append(float(m.group(2)))
            mious.append(float(m.group(3)))

print(f"epochs parsed: {len(epochs)} | best mIoU: {max(mious):.3f} (epoch {epochs[int(np.argmax(mious))]})")

fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(epochs, losses, "b-o", label="train loss")
ax1.set_xlabel("epoch"); ax1.set_ylabel("train loss", color="b"); ax1.tick_params(axis="y", labelcolor="b")
ax2 = ax1.twinx()
ax2.plot(epochs, mious, "r-s", label="val mIoU")
ax2.set_ylabel("val mIoU", color="r"); ax2.tick_params(axis="y", labelcolor="r")
ax2.axhline(max(mious), color="r", ls="--", alpha=0.4)
plt.title("Training curves")
fig.tight_layout(); plt.show()

## B) Predictions — RGB | ground truth | prediction

Loads the best model and runs it on **validation** tiles (same split used during training).

In [ ]:
# Load dataset, validation split and model
from train_seg import split_fids   # same stratified split (seed=42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ds = DisturbanceSegDataset(EMB_ROOT, TILES_ROOT, SHP_PATH)
_, val_fids = split_fids(ds)
val_idx = [i for i, s in enumerate(ds.samples) if s["fid"] in val_fids]
print(f"validation tiles: {len(val_idx)}")

model = SegDecoder(in_dim=768, num_classes=NUM_CLASSES).to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()
print(f"model loaded on {device}")

In [ ]:
# Discrete colormap for the classes (0..6)
COLORS = ["#000000", "#e41a1c", "#377eb8", "#4daf4a", "#984ea3", "#ff7f00", "#ffd92f"]
cmap = ListedColormap(COLORS[:NUM_CLASSES])
norm = BoundaryNorm(np.arange(NUM_CLASSES + 1) - 0.5, NUM_CLASSES)

from matplotlib.patches import Patch
legend = [Patch(facecolor=COLORS[c], label=f"{c}: {ID_TO_CLASS[c]}") for c in range(NUM_CLASSES)]

def show_prediction(i):
    s = ds.samples[i]
    x, y = ds[i]
    with torch.no_grad():
        pred = model(x[None].to(device)).argmax(1)[0].cpu().numpy()

    tif = f"{TILES_ROOT}/s2_l2a/fid_{s['fid']}/{s['window']}/{s['s2_id']}/{s['tile']}.tif"
    with rasterio.open(tif) as src:
        rgb = np.clip(src.read([4, 3, 2]).transpose(1, 2, 0) / 3000.0, 0, 1)

    fig, ax = plt.subplots(1, 3, figsize=(13, 4.5))
    ax[0].imshow(rgb); ax[0].set_title(f"RGB - fid {s['fid']} ({s['window']})")
    ax[1].imshow(y.numpy(), cmap=cmap, norm=norm); ax[1].set_title("ground truth")
    ax[2].imshow(pred, cmap=cmap, norm=norm); ax[2].set_title("prediction")
    for a in ax: a.axis("off")
    fig.legend(handles=legend, loc="lower center", ncol=4, fontsize=8, bbox_to_anchor=(0.5, -0.08))
    plt.tight_layout(); plt.show()

# show 6 random validation tiles
random.seed(0)
for i in random.sample(val_idx, min(6, len(val_idx))):
    show_prediction(i)